In [2]:
import bz2
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter, defaultdict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from wordcloud import WordCloud

import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\skadi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
reviews = []

file_path = "train.ft.txt.bz2"

with bz2.open(file_path, "rt", encoding="utf-8") as file:

    # Load first 50,000 reviews
    for i, line in enumerate(file):

        if i == 50000:
            break

        label, text = line.split(' ', 1)

        reviews.append([label, text])

df = pd.DataFrame(reviews, columns=["label", "review"])

print(df.head())
print("\nDataset Shape:", df.shape)


        label                                             review
0  __label__2  Stuning even for the non-gamer: This sound tra...
1  __label__2  The best soundtrack ever to anything.: I'm rea...
2  __label__2  Amazing!: This soundtrack is my favorite music...
3  __label__2  Excellent Soundtrack: I truly like this soundt...
4  __label__2  Remember, Pull Your Jaw Off The Floor After He...

Dataset Shape: (50000, 2)


In [8]:
# Review Length
df["char_length"] = df["review"].apply(len)

# Word Count
df["word_count"] = df["review"].apply(lambda x: len(x.split()))

# Empty Reviews
empty_reviews = (df["review"].str.strip() == "").sum()

# Duplicate Reviews
duplicate_reviews = df["review"].duplicated().sum()

print("Total Reviews:", len(df))
print("Empty Reviews:", empty_reviews)
print("Duplicate Reviews:", duplicate_reviews)

print("\nAverage Character Length:", df["char_length"].mean())
print("Average Word Count:", df["word_count"].mean())

Total Reviews: 50000
Empty Reviews: 0
Duplicate Reviews: 0

Average Character Length: 442.0161
Average Word Count: 80.17758


In [9]:
def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove Emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove Numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation/special chars
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [10]:
df["clean_review"] = df["review"].apply(clean_text)

print(df[["review", "clean_review"]].head())

                                              review  \
0  Stuning even for the non-gamer: This sound tra...   
1  The best soundtrack ever to anything.: I'm rea...   
2  Amazing!: This soundtrack is my favorite music...   
3  Excellent Soundtrack: I truly like this soundt...   
4  Remember, Pull Your Jaw Off The Floor After He...   

                                        clean_review  
0  stuning even for the non gamer this sound trac...  
1  the best soundtrack ever to anything i m readi...  
2  amazing this soundtrack is my favorite music o...  
3  excellent soundtrack i truly like this soundtr...  
4  remember pull your jaw off the floor after hea...  


In [11]:
stop_words = set(stopwords.words('english'))

def tokenize_text(text):

    tokens = word_tokenize(text)

    return tokens

df["tokens"] = df["clean_review"].apply(tokenize_text)

print(df["tokens"].head())

0    [stuning, even, for, the, non, gamer, this, so...
1    [the, best, soundtrack, ever, to, anything, i,...
2    [amazing, this, soundtrack, is, my, favorite, ...
3    [excellent, soundtrack, i, truly, like, this, ...
4    [remember, pull, your, jaw, off, the, floor, a...
Name: tokens, dtype: object


In [12]:
audit_results = []

for tokens in df["tokens"]:

    total_tokens = len(tokens)

    stopword_count = sum(
        1 for token in tokens
        if token in stop_words
    )

    informative_tokens = total_tokens - stopword_count

    stopword_ratio = (
        stopword_count / total_tokens
        if total_tokens > 0 else 0
    )

    informative_ratio = (
        informative_tokens / total_tokens
        if total_tokens > 0 else 0
    )

    audit_results.append([
        total_tokens,
        stopword_count,
        informative_tokens,
        stopword_ratio,
        informative_ratio
    ])

audit_df = pd.DataFrame(
    audit_results,
    columns=[
        "total_tokens",
        "stopword_tokens",
        "informative_tokens",
        "stopword_ratio",
        "informative_ratio"
    ]
)

print(audit_df.head())

   total_tokens  stopword_tokens  informative_tokens  stopword_ratio  \
0            80               36                  44        0.450000   
1           102               56                  46        0.549020   
2           135               67                  68        0.496296   
3           122               47                  75        0.385246   
4            90               44                  46        0.488889   

   informative_ratio  
0           0.550000  
1           0.450980  
2           0.503704  
3           0.614754  
4           0.511111  


In [13]:
bigram_counter = Counter()

for tokens in df["tokens"]:

    filtered_tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    bigrams = ngrams(filtered_tokens, 2)

    bigram_counter.update(bigrams)

top_bigrams = bigram_counter.most_common(20)

print("\nTop 20 Bigrams:\n")

for bigram, count in top_bigrams:
    print(bigram, ":", count)


Top 20 Bigrams:

('read', 'book') : 2035
('one', 'best') : 1072
('waste', 'money') : 1028
('year', 'old') : 1021
('would', 'recommend') : 990
('waste', 'time') : 908
('great', 'book') : 883
('much', 'better') : 819
('highly', 'recommend') : 772
('years', 'ago') : 769
('book', 'read') : 746
('first', 'time') : 719
('good', 'book') : 710
('reading', 'book') : 696
('ever', 'read') : 659
('recommend', 'book') : 613
('even', 'though') : 609
('great', 'movie') : 563
('long', 'time') : 536
('well', 'written') : 531


In [14]:
trigram_counter = Counter()

for tokens in df["tokens"]:

    filtered_tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    trigrams = ngrams(filtered_tokens, 3)

    trigram_counter.update(trigrams)

top_trigrams = trigram_counter.most_common(20)

print("\nTop 20 Trigrams:\n")

for trigram, count in top_trigrams:
    print(trigram, ":", count)


Top 20 Trigrams:

('book', 'ever', 'read') : 281
('waste', 'time', 'money') : 206
('books', 'ever', 'read') : 175
('would', 'recommend', 'book') : 164
('recommend', 'book', 'anyone') : 157
('would', 'recommend', 'anyone') : 154
('worst', 'movie', 'ever') : 142
('one', 'best', 'books') : 137
('worst', 'book', 'ever') : 125
('best', 'book', 'ever') : 124
('highly', 'recommend', 'book') : 123
('year', 'old', 'son') : 116
('movie', 'ever', 'seen') : 116
('would', 'highly', 'recommend') : 110
('pampers', 'baby', 'dry') : 106
('best', 'books', 'ever') : 92
('one', 'worst', 'movies') : 88
('year', 'old', 'daughter') : 86
('movies', 'ever', 'seen') : 82
('movie', 'ever', 'made') : 78


In [16]:
def detect_quality_issues(text):

    issues = []

    if re.search(r'https?://\S+', text):
        issues.append("URL")

    if re.search(r'\S+@\S+', text):
        issues.append("EMAIL")

    if re.search(r'<.*?>', text):
        issues.append("HTML")

    if len(text.split()) < 3:
        issues.append("TOO_SHORT")

    if len(text.split()) > 300:
        issues.append("TOO_LONG")

    return ", ".join(issues)

df["quality_issues"] = df["review"].apply(
    detect_quality_issues
)

print(df[["review", "quality_issues"]].head())

                                              review quality_issues
0  Stuning even for the non-gamer: This sound tra...               
1  The best soundtrack ever to anything.: I'm rea...               
2  Amazing!: This soundtrack is my favorite music...               
3  Excellent Soundtrack: I truly like this soundt...               
4  Remember, Pull Your Jaw Off The Floor After He...               


In [17]:
df.to_csv("cleaned_reviews.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
